In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# =========================================================
# AEROFLUX - FINAL GOLD LAYER
# =========================================================
#
# SOURCE
# /Volumes/aeroflux/silver_layer/trajectory-enriched
#
# GOLD OUTPUTS
#
# /Volumes/aeroflux/gold_layer/gold_4d_trajectory
# /Volumes/aeroflux/gold_layer/gold_h3_traffic
# /Volumes/aeroflux/gold_layer/gold_h3_flows
# /Volumes/aeroflux/gold_layer/gold_airport_activity
# /Volumes/aeroflux/gold_layer/gold_holding_events
# /Volumes/aeroflux/gold_layer/gold_h3_congestion
#
# =========================================================


# =========================================================
# 1. PATHS
# =========================================================

SOURCE_PATH = (
    "/Volumes/aeroflux/silver_layer/trajectory-enrichment"
)

GOLD_4D_PATH = (
    "/Volumes/aeroflux/gold_layer/gold_4d_trajectory"
)

GOLD_H3_TRAFFIC_PATH = (
    "/Volumes/aeroflux/gold_layer/gold_h3_traffic"
)

GOLD_H3_FLOWS_PATH = (
    "/Volumes/aeroflux/gold_layer/gold_h3_flows"
)

GOLD_AIRPORT_PATH = (
    "/Volumes/aeroflux/gold_layer/gold_airport_activity"
)

GOLD_HOLDING_PATH = (
    "/Volumes/aeroflux/gold_layer/gold_holding_events"
)

GOLD_CONGESTION_PATH = (
    "/Volumes/aeroflux/gold_layer/gold_h3_congestion"
)


# =========================================================
# 2. CONFIGURATION
# =========================================================

EARTH_RADIUS_KM = 6371.0

TRAFFIC_WINDOW = "5 minutes"
AIRPORT_WINDOW = "1 hour"
CONGESTION_WINDOW = "1 hour"

AIRPORT_RADIUS_KM = 50.0

MIN_HOLDING_DURATION_MINUTES = 15.0
MIN_HOLDING_OBSERVATIONS = 8

GOLD_VERSION = "gold-v3"


# =========================================================
# 3. READ TRAJECTORY SILVER
# =========================================================

df = (
    spark.read
    .format("delta")
    .load(SOURCE_PATH)
)


# =========================================================
# 4. SCHEMA VALIDATION
# =========================================================

required_columns = [
    "observation_id",
    "icao24",
    "event_time",

    "latitude",
    "longitude",

    "baro_altitude_m",
    "velocity_mps",
    "true_track",
    "vertical_rate_mps",

    "h3_res_6",
    "h3_res_8",
    "h3_res_10",
    "h3_cell",

    "trajectory_step_status",
    "trajectory_step_valid",

    "step_distance_km",
    "derived_speed_mps",

    "speed_difference_mps",
    "speed_error_ratio",

    "acceleration_mps2",
    "altitude_delta_m",
    "vertical_acceleration_mps2",

    "heading_change_deg",
    "turn_rate_deg_per_sec",

    "h3_transition",
    "h3_from_res_8",
    "h3_to_res_8",
    "h3_grid_distance",

    "time_delta_seconds",
    "telemetry_gap",

    "flight_session_id",
    "trajectory_segment_id",

    "segment_path_length_km",
    "segment_net_displacement_km",
    "segment_trajectory_efficiency",
]


missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "Missing required trajectory columns: "
        + ", ".join(missing_columns)
    )


# =========================================================
# 5. OPTIONAL on_ground
# =========================================================

if "on_ground" not in df.columns:

    df = df.withColumn(
        "on_ground",
        F.lit(None).cast("boolean")
    )


# =========================================================
# 6. BASIC NORMALIZATION
# =========================================================

df = (
    df

    .withColumn(
        "icao24",
        F.lower(
            F.trim(
                F.col("icao24")
            )
        )
    )

    .withColumn(
        "event_date",
        F.to_date(
            "event_time"
        )
    )

    .withColumn(
        "gold_version",
        F.lit(GOLD_VERSION)
    )
)


# =========================================================
# =========================================================
# GOLD 1
# 4D TRAJECTORY
# =========================================================
#
# Visualization source for the future 4D Digital Twin.
#
# 4D:
# latitude
# longitude
# altitude
# time
#
# trajectory_segment_id allows the visualization layer
# to break lines across telemetry discontinuities.
# =========================================================


segment_point_window = (
    Window
    .partitionBy(
        "icao24",
        "trajectory_segment_id"
    )
    .orderBy(
        F.col("event_time").asc(),
        F.col("observation_id").asc()
    )
)


gold_4d = (
    df

    .withColumn(
        "segment_point_index",
        F.row_number().over(
            segment_point_window
        )
    )

    .withColumn(
        "trajectory_line_break",
        (
            F.col("segment_point_index") == 1
        )
        |
        (
            ~F.col("trajectory_step_valid")
        )
    )

    .select(
        "observation_id",
        "icao24",

        "event_time",
        "event_date",

        "latitude",
        "longitude",
        "baro_altitude_m",

        "velocity_mps",
        "derived_speed_mps",

        "true_track",
        "vertical_rate_mps",

        "acceleration_mps2",
        "altitude_delta_m",
        "vertical_acceleration_mps2",

        "heading_change_deg",
        "turn_rate_deg_per_sec",

        "h3_res_6",
        "h3_res_8",
        "h3_res_10",
        "h3_cell",

        "flight_session_id",
        "trajectory_segment_id",
        "segment_point_index",

        "trajectory_step_status",
        "trajectory_step_valid",
        "trajectory_line_break",

        "time_delta_seconds",
        "telemetry_gap",

        "gold_version"
    )
)


(
    gold_4d
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .save(
        GOLD_4D_PATH
    )
)


# =========================================================
# =========================================================
# GOLD 2
# H3 TRAFFIC
# =========================================================
#
# Grain:
# H3 R8 × 5-minute window
#
# =========================================================


spatial_df = (
    df
    .filter(
        F.col("h3_res_8").isNotNull()
    )
)


traffic = (
    spatial_df

    .groupBy(
        F.window(
            "event_time",
            TRAFFIC_WINDOW
        ).alias("traffic_window"),

        "h3_res_8"
    )

    .agg(

        F.countDistinct(
            "icao24"
        ).alias(
            "unique_aircraft_count"
        ),

        F.count(
            "*"
        ).alias(
            "observation_count"
        ),

        F.sum(
            F.when(
                F.col("trajectory_step_valid"),
                1
            )
            .otherwise(0)
        ).alias(
            "valid_transition_count"
        ),

        F.avg(
            "baro_altitude_m"
        ).alias(
            "mean_altitude_m"
        ),

        F.stddev(
            "baro_altitude_m"
        ).alias(
            "altitude_stddev_m"
        ),

        F.avg(
            "velocity_mps"
        ).alias(
            "mean_velocity_mps"
        ),

        F.stddev(
            "velocity_mps"
        ).alias(
            "velocity_stddev_mps"
        ),

        F.expr(
            "percentile_approx("
            "velocity_mps, 0.95"
            ")"
        ).alias(
            "p95_velocity_mps"
        ),

        F.sum(
            F.when(
                F.col("trajectory_step_valid"),
                F.least(
                    F.lit(120.0),
                    F.greatest(
                        F.lit(0.0),
                        F.coalesce(
                            F.col(
                                "time_delta_seconds"
                            ),
                            F.lit(0.0)
                        )
                    )
                )
            )
            .otherwise(0.0)
        ).alias(
            "observed_transition_seconds"
        )
    )

    .withColumn(
        "window_start",
        F.col(
            "traffic_window.start"
        )
    )

    .withColumn(
        "window_end",
        F.col(
            "traffic_window.end"
        )
    )

    .drop(
        "traffic_window"
    )
)


# =========================================================
# H3 INFLOW
# =========================================================

h3_inflow = (
    spatial_df

    .filter(
        F.col("trajectory_step_valid")
        &
        F.col("h3_transition")
        &
        F.col("h3_to_res_8").isNotNull()
    )

    .groupBy(
        F.window(
            "event_time",
            TRAFFIC_WINDOW
        ).alias("flow_window"),

        F.col("h3_to_res_8").alias(
            "h3_res_8"
        )
    )

    .agg(
        F.countDistinct(
            "icao24"
        ).alias(
            "inflow_unique_aircraft"
        ),

        F.count(
            "*"
        ).alias(
            "inflow_transition_count"
        )
    )

    .withColumn(
        "window_start",
        F.col(
            "flow_window.start"
        )
    )

    .withColumn(
        "window_end",
        F.col(
            "flow_window.end"
        )
    )

    .drop(
        "flow_window"
    )
)


# =========================================================
# H3 OUTFLOW
# =========================================================

h3_outflow = (
    spatial_df

    .filter(
        F.col("trajectory_step_valid")
        &
        F.col("h3_transition")
        &
        F.col("h3_from_res_8").isNotNull()
    )

    .groupBy(
        F.window(
            "event_time",
            TRAFFIC_WINDOW
        ).alias("flow_window"),

        F.col("h3_from_res_8").alias(
            "h3_res_8"
        )
    )

    .agg(
        F.countDistinct(
            "icao24"
        ).alias(
            "outflow_unique_aircraft"
        ),

        F.count(
            "*"
        ).alias(
            "outflow_transition_count"
        )
    )

    .withColumn(
        "window_start",
        F.col(
            "flow_window.start"
        )
    )

    .withColumn(
        "window_end",
        F.col(
            "flow_window.end"
        )
    )

    .drop(
        "flow_window"
    )
)


# =========================================================
# FINAL H3 TRAFFIC
# =========================================================

gold_h3_traffic = (
    traffic

    .join(
        h3_inflow,
        [
            "h3_res_8",
            "window_start",
            "window_end"
        ],
        "left"
    )

    .join(
        h3_outflow,
        [
            "h3_res_8",
            "window_start",
            "window_end"
        ],
        "left"
    )

    .fillna(
        0,
        [
            "inflow_unique_aircraft",
            "inflow_transition_count",
            "outflow_unique_aircraft",
            "outflow_transition_count"
        ]
    )

    .withColumn(
        "net_aircraft_flow",
        F.col(
            "inflow_unique_aircraft"
        )
        -
        F.col(
            "outflow_unique_aircraft"
        )
    )

    .withColumn(
        "gold_version",
        F.lit(GOLD_VERSION)
    )
)


(
    gold_h3_traffic
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .save(
        GOLD_H3_TRAFFIC_PATH
    )
)


# =========================================================
# =========================================================
# GOLD 3
# H3 FLOWS
# =========================================================
#
# Grain:
#
# H3 R8 → H3 R8 × 5-minute window
#
# =========================================================


gold_h3_flows = (
    spatial_df

    .filter(
        F.col("trajectory_step_valid")
        &
        F.col("h3_transition")
        &
        F.col("h3_from_res_8").isNotNull()
        &
        F.col("h3_to_res_8").isNotNull()
    )

    .groupBy(
        F.window(
            "event_time",
            TRAFFIC_WINDOW
        ).alias("flow_window"),

        "h3_from_res_8",
        "h3_to_res_8"
    )

    .agg(

        F.countDistinct(
            "icao24"
        ).alias(
            "unique_aircraft"
        ),

        F.count(
            "*"
        ).alias(
            "transition_count"
        ),

        F.avg(
            "step_distance_km"
        ).alias(
            "mean_step_distance_km"
        ),

        F.avg(
            "time_delta_seconds"
        ).alias(
            "mean_transition_time_seconds"
        ),

        F.avg(
            "derived_speed_mps"
        ).alias(
            "mean_derived_speed_mps"
        )
    )

    .withColumn(
        "window_start",
        F.col(
            "flow_window.start"
        )
    )

    .withColumn(
        "window_end",
        F.col(
            "flow_window.end"
        )
    )

    .withColumn(
        "h3_grid_distance",
        F.expr(
            "h3_distance("
            "h3_from_res_8, "
            "h3_to_res_8"
            ")"
        )
    )

    .withColumn(
        "gold_version",
        F.lit(GOLD_VERSION)
    )

    .drop(
        "flow_window"
    )
)


(
    gold_h3_flows
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .save(
        GOLD_H3_FLOWS_PATH
    )
)


# =========================================================
# =========================================================
# GOLD 4
# AIRPORT ACTIVITY
# =========================================================
#
# CONFIGURED AIRPORT REFERENCE
#
# IMPORTANT:
# This is a six-airport analytical reference set.
#
# 50 km crossing = airport analysis-region entry/exit.
# It is NOT a confirmed arrival/departure.
# =========================================================


AIRPORTS = [
    ("JFK", 40.6413, -73.7781),
    ("LHR", 51.4700, -0.4543),
    ("CDG", 49.0097, 2.5479),
    ("DXB", 25.2532, 55.3657),
    ("DEL", 28.5562, 77.1000),
    ("BOM", 19.0896, 72.8656),
]


airport_df = spark.createDataFrame(
    AIRPORTS,
    [
        "airport",
        "airport_lat",
        "airport_lon"
    ]
)


# =========================================================
# AIRCRAFT × AIRPORT
# =========================================================

airport_observations = (
    df

    .filter(
        F.col("latitude").isNotNull()
        &
        F.col("longitude").isNotNull()
    )

    .crossJoin(
        F.broadcast(
            airport_df
        )
    )
)


# =========================================================
# NATIVE SPARK HAVERSINE
# =========================================================

lat1 = F.radians(
    F.col("latitude")
)

lon1 = F.radians(
    F.col("longitude")
)

lat2 = F.radians(
    F.col("airport_lat")
)

lon2 = F.radians(
    F.col("airport_lon")
)

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    F.pow(
        F.sin(dlat / 2),
        2
    )
    +
    F.cos(lat1)
    * F.cos(lat2)
    * F.pow(
        F.sin(dlon / 2),
        2
    )
)

a_safe = F.least(
    F.lit(1.0),
    F.greatest(
        F.lit(0.0),
        a
    )
)


airport_distance_expr = (
    F.lit(2.0)
    * F.lit(EARTH_RADIUS_KM)
    * F.asin(
        F.sqrt(a_safe)
    )
)


airport_observations = (
    airport_observations

    .withColumn(
        "distance_to_airport_km",
        airport_distance_expr
    )

    .withColumn(
        "inside_airport_radius",
        (
            F.col(
                "distance_to_airport_km"
            )
            <=
            AIRPORT_RADIUS_KM
        )
    )
)


# =========================================================
# PREVIOUS AIRPORT DISTANCE
# =========================================================

airport_window = (
    Window
    .partitionBy(
        "icao24",
        "airport"
    )
    .orderBy(
        F.col("event_time").asc(),
        F.col("observation_id").asc()
    )
)


airport_observations = (
    airport_observations

    .withColumn(
        "previous_airport_distance_km",
        F.lag(
            "distance_to_airport_km"
        ).over(
            airport_window
        )
    )

    .withColumn(
        "previous_inside_airport_radius",
        F.lag(
            "inside_airport_radius"
        ).over(
            airport_window
        )
    )
)


# =========================================================
# AIRPORT ENTRY / EXIT
#
# Only valid trajectory transitions can generate crossings.
# =========================================================

airport_observations = (
    airport_observations

    .withColumn(
        "airport_region_entry",
        (
            F.col("trajectory_step_valid")
            &
            F.col("inside_airport_radius")
            &
            F.col(
                "previous_inside_airport_radius"
            ).isNotNull()
            &
            ~F.col(
                "previous_inside_airport_radius"
            )
        )
    )

    .withColumn(
        "airport_region_exit",
        (
            F.col("trajectory_step_valid")
            &
            ~F.col("inside_airport_radius")
            &
            F.col(
                "previous_inside_airport_radius"
            ).isNotNull()
            &
            F.col(
                "previous_inside_airport_radius"
            )
        )
    )
)


# =========================================================
# NEAREST AIRPORT PER OBSERVATION
# =========================================================

nearest_airport_window = (
    Window
    .partitionBy(
        "observation_id"
    )
    .orderBy(
        F.col(
            "distance_to_airport_km"
        ).asc()
    )
)


nearest_airport = (
    airport_observations

    .withColumn(
        "airport_rank",
        F.row_number().over(
            nearest_airport_window
        )
    )

    .filter(
        F.col("airport_rank") == 1
    )

    .select(
        "observation_id",
        F.col("airport").alias(
            "nearest_airport"
        ),
        F.col(
            "distance_to_airport_km"
        ).alias(
            "distance_to_nearest_airport_km"
        )
    )
)


# =========================================================
# AIRPORT HOURLY ACTIVITY
# =========================================================

gold_airport_activity = (
    airport_observations

    .groupBy(
        F.window(
            "event_time",
            AIRPORT_WINDOW
        ).alias("airport_window"),

        "airport"
    )

    .agg(

        F.countDistinct(
            F.when(
                F.col("inside_airport_radius"),
                F.col("icao24")
            )
        ).alias(
            "unique_aircraft"
        ),

        F.sum(
            F.when(
                F.col("airport_region_entry"),
                1
            ).otherwise(0)
        ).alias(
            "airport_region_entries"
        ),

        F.sum(
            F.when(
                F.col("airport_region_exit"),
                1
            ).otherwise(0)
        ).alias(
            "airport_region_exits"
        ),

        F.avg(
            F.when(
                F.col("inside_airport_radius"),
                F.col(
                    "distance_to_airport_km"
                )
            )
        ).alias(
            "mean_distance_to_airport_km"
        ),

        F.expr(
            "percentile_approx("
            "CASE WHEN inside_airport_radius "
            "THEN distance_to_airport_km END, "
            "0.95)"
        ).alias(
            "p95_distance_to_airport_km"
        ),

        F.avg(
            F.when(
                F.col("inside_airport_radius"),
                F.col("baro_altitude_m")
            )
        ).alias(
            "mean_altitude_m"
        ),

        F.avg(
            F.when(
                F.col("inside_airport_radius"),
                F.col("velocity_mps")
            )
        ).alias(
            "mean_velocity_mps"
        )
    )

    .withColumn(
        "window_start",
        F.col(
            "airport_window.start"
        )
    )

    .withColumn(
        "window_end",
        F.col(
            "airport_window.end"
        )
    )

    .withColumn(
        "airport_analysis_radius_km",
        F.lit(
            AIRPORT_RADIUS_KM
        )
    )

    .withColumn(
        "airport_reference_version",
        F.lit(
            "configured-airports-v1"
        )
    )

    .withColumn(
        "gold_version",
        F.lit(GOLD_VERSION)
    )

    .drop(
        "airport_window"
    )
)


(
    gold_airport_activity
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .save(
        GOLD_AIRPORT_PATH
    )
)


# =========================================================
# ADD NEAREST AIRPORT TO 4D DATA
# =========================================================

gold_4d_with_airport = (
    gold_4d

    .drop(
        "gold_version"
    )

    .join(
        nearest_airport,
        "observation_id",
        "left"
    )

    .withColumn(
        "gold_version",
        F.lit(GOLD_VERSION)
    )
)


(
    gold_4d_with_airport
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .save(
        GOLD_4D_PATH
    )
)


# =========================================================
# =========================================================
# GOLD 5
# HOLDING EVENTS
# =========================================================
#
# V2 trajectory-based holding candidate.
#
# Criteria:
#
#   ≥ 15 minutes
#   ≥ 8 observations
#   within 50 km airport region
#   low trajectory efficiency
#   repeated heading changes
#   H3 revisitation
#   airborne evidence
#
# This is an inferred/observed event, NOT an official
# aviation-certified holding classification.
# =========================================================


holding_points = (
    airport_observations

    .filter(
        F.col(
            "inside_airport_radius"
        )
    )

    .withColumn(
        "local_step_valid",
        (
            F.col("trajectory_step_valid")
            &
            F.col(
                "previous_airport_distance_km"
            ).isNotNull()
            &
            (
                F.col(
                    "previous_airport_distance_km"
                )
                <=
                AIRPORT_RADIUS_KM
            )
        )
    )
)


# =========================================================
# HOLDING SEGMENT FEATURES
# =========================================================

holding_segments = (
    holding_points

    .groupBy(
        "icao24",
        "airport",
        "trajectory_segment_id"
    )

    .agg(

        F.min(
            "event_time"
        ).alias(
            "holding_start"
        ),

        F.max(
            "event_time"
        ).alias(
            "holding_end"
        ),

        F.count(
            "*"
        ).alias(
            "observation_count"
        ),

        F.sum(
            F.when(
                F.col("local_step_valid"),
                1
            ).otherwise(0)
        ).alias(
            "valid_local_steps"
        ),

        F.countDistinct(
            "h3_res_8"
        ).alias(
            "unique_h3_cells"
        ),

        F.sum(
            F.when(
                F.col("local_step_valid"),
                F.coalesce(
                    F.col("step_distance_km"),
                    F.lit(0.0)
                )
            ).otherwise(0.0)
        ).alias(
            "path_length_km"
        ),

        F.min(
            "distance_to_airport_km"
        ).alias(
            "minimum_airport_distance_km"
        ),

        F.avg(
            "baro_altitude_m"
        ).alias(
            "mean_altitude_m"
        ),

        F.stddev(
            "baro_altitude_m"
        ).alias(
            "altitude_stddev_m"
        ),

        F.avg(
            F.abs(
                F.col("vertical_rate_mps")
            )
        ).alias(
            "mean_abs_vertical_rate_mps"
        ),

        F.sum(
            F.when(
                F.col("heading_change_deg") >= 30,
                1
            ).otherwise(0)
        ).alias(
            "large_heading_change_count"
        ),

        F.sum(
            F.when(
                F.col("heading_change_deg") >= 60,
                1
            ).otherwise(0)
        ).alias(
            "large_turn_count"
        ),

        F.sum(
            F.when(
                F.col("on_ground") == True,
                1
            ).otherwise(0)
        ).alias(
            "on_ground_true_count"
        ),

        F.sum(
            F.when(
                F.col("on_ground") == False,
                1
            ).otherwise(0)
        ).alias(
            "on_ground_false_count"
        ),

        # H3 cell representing the beginning of the
        # observed holding-like segment.
        F.min_by(
            "h3_res_8",
            "event_time"
        ).alias(
            "representative_h3_res_8"
        ),

        # Start/end points must correspond to the actual
        # local valid path, not to an excluded crossing.
        F.min_by(
            "previous_latitude",
            "event_time"
        ).alias(
            "start_latitude"
        ),

        F.min_by(
            "previous_longitude",
            "event_time"
        ).alias(
            "start_longitude"
        ),

        F.max_by(
            "latitude",
            "event_time"
        ).alias(
            "end_latitude"
        ),

        F.max_by(
            "longitude",
            "event_time"
        ).alias(
            "end_longitude"
        )
    )
)


# =========================================================
# HOLDING DURATION
# =========================================================

holding_segments = holding_segments.withColumn(
    "duration_minutes",
    (
        F.col("holding_end").cast("long")
        -
        F.col("holding_start").cast("long")
    )
    / 60.0
)


# =========================================================
# H3 REVISIT RATIO
# =========================================================

holding_segments = holding_segments.withColumn(
    "h3_revisit_ratio",
    F.when(
        F.col("observation_count") > 0,
        (
            1.0
            -
            (
                F.col("unique_h3_cells")
                /
                F.col("observation_count")
            )
        )
    )
)


# =========================================================
# LOCAL NET DISPLACEMENT
# =========================================================

h_lat1 = F.radians(
    F.col("start_latitude")
)

h_lon1 = F.radians(
    F.col("start_longitude")
)

h_lat2 = F.radians(
    F.col("end_latitude")
)

h_lon2 = F.radians(
    F.col("end_longitude")
)

h_dlat = h_lat2 - h_lat1
h_dlon = h_lon2 - h_lon1

h_a = (
    F.pow(
        F.sin(h_dlat / 2),
        2
    )
    +
    F.cos(h_lat1)
    * F.cos(h_lat2)
    * F.pow(
        F.sin(h_dlon / 2),
        2
    )
)


holding_segments = holding_segments.withColumn(
    "net_displacement_km",
    F.lit(2.0)
    * F.lit(EARTH_RADIUS_KM)
    * F.asin(
        F.sqrt(
            F.least(
                F.lit(1.0),
                F.greatest(
                    F.lit(0.0),
                    h_a
                )
            )
        )
    )
)


# =========================================================
# TRAJECTORY EFFICIENCY
# =========================================================

holding_segments = holding_segments.withColumn(
    "trajectory_efficiency",
    F.when(
        F.col("path_length_km") > 0,
        F.col("net_displacement_km")
        /
        F.col("path_length_km")
    )
)


# =========================================================
# AIRBORNE STATUS
# =========================================================

holding_segments = holding_segments.withColumn(
    "airborne_status",
    F.when(
        (
            F.col("on_ground_true_count") > 0
        )
        &
        (
            F.col("on_ground_false_count") == 0
        ),
        "GROUND_ONLY"
    )
    .when(
        F.col("on_ground_false_count") > 0,
        "AIRBORNE_EVIDENCE"
    )
    .otherwise(
        "UNKNOWN"
    )
)


# =========================================================
# HOLDING CANDIDATE
# =========================================================

holding_segments = holding_segments.withColumn(
    "holding_candidate",
    (
        (F.col("duration_minutes")
         >= MIN_HOLDING_DURATION_MINUTES)

        &

        (F.col("observation_count")
         >= MIN_HOLDING_OBSERVATIONS)

        &

        (F.col("valid_local_steps")
         >= 2)

        &

        (F.col("minimum_airport_distance_km")
         <= AIRPORT_RADIUS_KM)

        &

        F.col("trajectory_efficiency").isNotNull()

        &

        (F.col("trajectory_efficiency")
         <= 0.60)

        &

        (F.col("large_heading_change_count")
         >= 4)

        &

        (F.col("h3_revisit_ratio")
         >= 0.10)

        &

        (
            F.col("airborne_status")
            != "GROUND_ONLY"
        )
    )
)


# =========================================================
# HOLDING SCORE
# =========================================================

proximity_score = (
    1.0
    -
    F.least(
        F.lit(1.0),
        F.col(
            "minimum_airport_distance_km"
        )
        /
        F.lit(AIRPORT_RADIUS_KM)
    )
)

dwell_score = F.least(
    F.lit(1.0),
    F.col("duration_minutes")
    /
    F.lit(30.0)
)

efficiency_score = (
    1.0
    -
    F.least(
        F.lit(1.0),
        F.col("trajectory_efficiency")
        /
        F.lit(0.60)
    )
)

turn_score = F.least(
    F.lit(1.0),
    F.col(
        "large_heading_change_count"
    )
    /
    F.lit(8.0)
)

revisit_score = F.least(
    F.lit(1.0),
    F.col("h3_revisit_ratio")
    /
    F.lit(0.50)
)

vertical_stability_score = (
    1.0
    -
    F.least(
        F.lit(1.0),
        F.coalesce(
            F.col(
                "mean_abs_vertical_rate_mps"
            ),
            F.lit(9999.0)
        )
        /
        F.lit(5.0)
    )
)


holding_segments = holding_segments.withColumn(
    "holding_score",
    (
        F.lit(0.15) * proximity_score
        +
        F.lit(0.15) * dwell_score
        +
        F.lit(0.20) * efficiency_score
        +
        F.lit(0.20) * turn_score
        +
        F.lit(0.15) * revisit_score
        +
        F.lit(0.15) * vertical_stability_score
    )
)


# =========================================================
# HOLDING CONFIDENCE
# =========================================================

holding_segments = holding_segments.withColumn(
    "holding_confidence",
    F.when(
        F.col("holding_score") >= 0.70,
        "HIGH"
    )
    .when(
        F.col("holding_score") >= 0.50,
        "MEDIUM"
    )
    .otherwise(
        "LOW"
    )
)


# =========================================================
# FINAL HOLDING EVENTS
# =========================================================

gold_holding_events = (
    holding_segments

    .filter(
        F.col("holding_candidate")
    )

    .withColumn(
        "holding_event_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("icao24"),
                F.col("airport"),
                F.col(
                    "holding_start"
                ).cast("long"),
                F.col(
                    "trajectory_segment_id"
                )
            ),
            256
        )
    )

    .withColumn(
        "detection_version",
        F.lit("holding-v2")
    )

    .select(
        "holding_event_id",

        "icao24",
        "airport",
        "trajectory_segment_id",
        "representative_h3_res_8",

        "holding_start",
        "holding_end",
        "duration_minutes",

        "minimum_airport_distance_km",

        "path_length_km",
        "net_displacement_km",
        "trajectory_efficiency",

        "mean_altitude_m",
        "altitude_stddev_m",

        "mean_abs_vertical_rate_mps",

        "large_heading_change_count",
        "large_turn_count",

        "unique_h3_cells",
        "h3_revisit_ratio",

        "observation_count",
        "valid_local_steps",

        "airborne_status",

        "holding_score",
        "holding_confidence",

        "detection_version"
    )
)


(
    gold_holding_events
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .save(
        GOLD_HOLDING_PATH
    )
)


# =========================================================
# =========================================================
# GOLD 6
# H3 CONGESTION
# =========================================================
#
# This is an observed traffic anomaly proxy.
#
# It does NOT represent official ATC capacity.
#
# A statistical baseline is considered supported only when
# an H3 cell + hour/day pattern has at least 3 observations
# and non-zero variance.
# =========================================================


h3_hourly = (
    spatial_df

    .groupBy(
        F.window(
            "event_time",
            CONGESTION_WINDOW
        ).alias(
            "congestion_window"
        ),

        "h3_res_8"
    )

    .agg(

        F.countDistinct(
            "icao24"
        ).alias(
            "unique_aircraft_count"
        ),

        F.count(
            "*"
        ).alias(
            "observation_count"
        ),

        F.avg(
            "velocity_mps"
        ).alias(
            "mean_velocity_mps"
        ),

        F.stddev(
            "velocity_mps"
        ).alias(
            "velocity_stddev_mps"
        ),

        F.avg(
            "baro_altitude_m"
        ).alias(
            "mean_altitude_m"
        )
    )

    .withColumn(
        "window_start",
        F.col(
            "congestion_window.start"
        )
    )

    .withColumn(
        "window_end",
        F.col(
            "congestion_window.end"
        )
    )

    .withColumn(
        "hour_of_day",
        F.hour(
            F.col("window_start")
        )
    )

    .withColumn(
        "day_of_week",
        F.dayofweek(
            F.col("window_start")
        )
    )

    .drop(
        "congestion_window"
    )
)


# =========================================================
# HISTORICAL BASELINE
# =========================================================
#
# Cell + day-of-week + hour
# =========================================================

h3_baseline = (
    h3_hourly

    .groupBy(
        "h3_res_8",
        "day_of_week",
        "hour_of_day"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "baseline_window_count"
        ),

        F.avg(
            "unique_aircraft_count"
        ).alias(
            "baseline_mean_aircraft"
        ),

        F.stddev(
            "unique_aircraft_count"
        ).alias(
            "baseline_std_aircraft"
        ),

        F.expr(
            "percentile_approx("
            "unique_aircraft_count, "
            "0.50)"
        ).alias(
            "baseline_median_aircraft"
        )
    )
)


# =========================================================
# JOIN BASELINE
# =========================================================

h3_congestion = (
    h3_hourly

    .join(
        h3_baseline,
        [
            "h3_res_8",
            "day_of_week",
            "hour_of_day"
        ],
        "left"
    )
)


# =========================================================
# BASELINE SUPPORT
# =========================================================

h3_congestion = h3_congestion.withColumn(
    "baseline_supported",
    (
        F.col(
            "baseline_window_count"
        )
        >= 3
    )
    &
    F.col(
        "baseline_std_aircraft"
    ).isNotNull()
    &
    (
        F.col(
            "baseline_std_aircraft"
        )
        > 0
    )
)


# =========================================================
# TRAFFIC Z-SCORE
# =========================================================

h3_congestion = h3_congestion.withColumn(
    "traffic_zscore",
    F.when(
        F.col("baseline_supported"),
        (
            F.col(
                "unique_aircraft_count"
            )
            -
            F.col(
                "baseline_mean_aircraft"
            )
        )
        /
        F.col(
            "baseline_std_aircraft"
        )
    )
)


# =========================================================
# TRAFFIC EXCESS RATIO
# =========================================================

h3_congestion = h3_congestion.withColumn(
    "traffic_excess_ratio",
    F.when(
        F.col(
            "baseline_mean_aircraft"
        ) > 0,
        F.col(
            "unique_aircraft_count"
        )
        /
        F.col(
            "baseline_mean_aircraft"
        )
    )
)


# =========================================================
# TRAFFIC PRESSURE
# =========================================================

h3_congestion = h3_congestion.withColumn(
    "traffic_pressure_score",
    F.when(
        F.col("baseline_supported"),
        F.least(
            F.lit(1.0),
            F.greatest(
                F.lit(0.0),
                (
                    F.col(
                        "traffic_zscore"
                    )
                    -
                    F.lit(1.0)
                )
                /
                F.lit(2.0)
            )
        )
    )
)


# =========================================================
# HOLDING EVENTS BY H3 + HOUR
# =========================================================

holding_activity = (
    gold_holding_events

    .withColumn(
        "window_start",
        F.date_trunc(
            "hour",
            F.col("holding_start")
        )
    )

    .groupBy(
        "representative_h3_res_8",
        "window_start"
    )

    .agg(
        F.countDistinct(
            "icao24"
        ).alias(
            "holding_aircraft_count"
        ),

        F.count(
            "*"
        ).alias(
            "holding_event_count"
        )
    )

    .withColumnRenamed(
        "representative_h3_res_8",
        "h3_res_8"
    )
)


# =========================================================
# JOIN HOLDING PRESSURE
# =========================================================

h3_congestion = (
    h3_congestion

    .join(
        holding_activity,
        [
            "h3_res_8",
            "window_start"
        ],
        "left"
    )

    .fillna(
        0,
        [
            "holding_aircraft_count",
            "holding_event_count"
        ]
    )
)


# =========================================================
# HOLDING PRESSURE
# =========================================================

h3_congestion = h3_congestion.withColumn(
    "holding_pressure_score",
    F.least(
        F.lit(1.0),
        F.col(
            "holding_event_count"
        )
        /
        F.lit(3.0)
    )
)


# =========================================================
# CONGESTION SCORE
#
# Only calculate when traffic baseline is supported.
# =========================================================

h3_congestion = h3_congestion.withColumn(
    "congestion_score",
    F.when(
        F.col("baseline_supported"),
        (
            F.lit(0.80)
            *
            F.coalesce(
                F.col(
                    "traffic_pressure_score"
                ),
                F.lit(0.0)
            )
        )
        +
        (
            F.lit(0.20)
            *
            F.col(
                "holding_pressure_score"
            )
        )
    )
)


# =========================================================
# CONGESTION SIGNAL
# =========================================================

h3_congestion = h3_congestion.withColumn(
    "congestion_signal",
    F.when(
        ~F.col("baseline_supported"),
        "INSUFFICIENT_BASELINE"
    )
    .when(
        F.col("traffic_zscore") >= 2,
        "HIGH"
    )
    .when(
        F.col("traffic_zscore") >= 1,
        "ELEVATED"
    )
    .otherwise(
        "NORMAL"
    )
)


# =========================================================
# FINAL CONGESTION GOLD
# =========================================================

gold_h3_congestion = (
    h3_congestion

    .withColumn(
        "congestion_definition",
        F.lit(
            "observed H3 traffic anomaly proxy based on "
            "historical traffic baseline and holding activity"
        )
    )

    .withColumn(
        "gold_version",
        F.lit(GOLD_VERSION)
    )

    .select(
        "h3_res_8",

        "window_start",
        "window_end",

        "day_of_week",
        "hour_of_day",

        "unique_aircraft_count",
        "observation_count",

        "mean_velocity_mps",
        "velocity_stddev_mps",
        "mean_altitude_m",

        "baseline_window_count",
        "baseline_supported",

        "baseline_mean_aircraft",
        "baseline_median_aircraft",
        "baseline_std_aircraft",

        "traffic_zscore",
        "traffic_excess_ratio",

        "holding_aircraft_count",
        "holding_event_count",

        "traffic_pressure_score",
        "holding_pressure_score",

        "congestion_score",
        "congestion_signal",

        "congestion_definition",
        "gold_version"
    )
)


(
    gold_h3_congestion
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .save(
        GOLD_CONGESTION_PATH
    )
)


# =========================================================
# FINAL VALIDATION
# =========================================================

gold_products = {
    "4D trajectory":
        GOLD_4D_PATH,

    "H3 traffic":
        GOLD_H3_TRAFFIC_PATH,

    "H3 flows":
        GOLD_H3_FLOWS_PATH,

    "Airport activity":
        GOLD_AIRPORT_PATH,

    "Holding events":
        GOLD_HOLDING_PATH,

    "H3 congestion":
        GOLD_CONGESTION_PATH,
}


print()
print("=" * 70)
print("AEROFLUX GOLD PRODUCTS")
print("=" * 70)

for name, path in gold_products.items():

    count = (
        spark.read
        .format("delta")
        .load(path)
        .count()
    )

    print(
        f"{name:<25} : {count:,} rows"
    )

print("=" * 70)


# =========================================================
# HOLDING SUMMARY
# =========================================================

holding_result = (
    spark.read
    .format("delta")
    .load(GOLD_HOLDING_PATH)
)

print()
print(
    "Holding events detected : "
    f"{holding_result.count():,}"
)


# =========================================================
# CONGESTION SIGNAL SUMMARY
# =========================================================

congestion_result = (
    spark.read
    .format("delta")
    .load(GOLD_CONGESTION_PATH)
)


display(
    congestion_result
    .groupBy(
        "congestion_signal"
    )
    .count()
    .orderBy(
        "congestion_signal"
    )
)


AEROFLUX GOLD PRODUCTS
4D trajectory             : 108,103 rows
H3 traffic                : 90,801 rows
H3 flows                  : 76,796 rows
Airport activity          : 144 rows
Holding events            : 0 rows
H3 congestion             : 89,643 rows

Holding events detected : 0


congestion_signal,count
INSUFFICIENT_BASELINE,89643
